⏱️ **Time required:** ~15 minutes | **Type:** Interactive tutorial

# Data Generation & AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/05_data_generation_ai.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/05_data_generation_ai.ipynb)

Generate synthetic test data with referential integrity across tables, inject edge cases, validate quarantine coverage, and auto-infer contracts from CSVs.

In [1]:
# Install lakelogic
!pip install -q lakelogic[polars,extraction-ocr,nlp]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

lakelogic v1.21.0 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb" # 'polars' , 'spark'

---
## 1. DataGenerator Basics — Synthetic Data From a Contract

**The Problem:** You need test data that matches your schema. Writing Faker scripts for every table is tedious and drifts out of sync with your contracts.

**The Solution:** `DataGenerator` reads your contract and generates realistic data — including controlled invalid rows for quarantine testing.

In [2]:
contract = s.write_contract(
    """
version: 1.0.0
dataset: test_users
model:
  fields:
    - name: user_id
      type: integer
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: age
      type: integer
      min: 18
      max: 120
    - name: country
      type: string
      accepted_values: [US, GB, DE, FR, JP]
    - name: status
      type: string
      accepted_values: [active, inactive, suspended]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: valid_age
      sql: "age BETWEEN 18 AND 120"
    - name: valid_country
      sql: "country IN ('US','GB','DE','FR','JP')"
""",
    "05_data_generation_ai_demo/users.yaml",
)

gen = ll.DataGenerator(contract)
df = gen.generate(rows=1000, invalid_ratio=0.10, output_format=ENGINE)

2026-04-28 06:50:18.167 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: test_users
2026-04-28 06:50:18.167 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 900 valid + 100 invalid = 1,000 total
2026-04-28 06:50:18.167 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:18.168 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:50:18.288 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 1,000 records built
2026-04-28 06:50:18.288 | INFO     | lakelogic.core.generator:generate:3504 -    Test cases : 200 across 7 categories
2026-04-28 06:50:18.289 | INFO     | lakelogic.core.generator:generate:3506 -      NOT_NULL_VIOLATION               64 injections
2026-04-28 06:50:18.289 | INFO     | lakelogic.core.generator:generate:3506 -      EMPTY_STRING    

In [3]:
# The Proof
print(f"Generated {len(df)} rows with ~10% intentionally invalid")
display(df.head(10))

Generated 1000 rows with ~10% intentionally invalid


user_id,email,age,country,status,_is_invalid,_test_case_types
i64,str,i64,str,str,bool,str
5257,"""charles91@example.com""",25,"""FR""","""active""",false,null
3984,"""cantutiffany@example.com""",40,"""FR""","""active""",false,null
4239,"""angelaweiss@example.org""",25,"""FR""","""inactive""",false,null
6610,"""michael88@example.com""",49,"""DE""","""active""",false,null
9533,"""bakerjackie@example.com""",36,"""US""","""""",true,"""EMPTY_STRING"""
-92,"""romerogerald@example.org""",25,null,"""inactive""",true,"""NOT_NULL_VIOLATION,RANGE_VIOLATION"""
600,"""kaitlynrodriguez@example.org""",31,"""GB""","""active""",false,null
6418,"""jeremyharvey@example.org""",30,"""FR""","""active""",true,null
null,"""herrerabrian@example.net""",46,"""JP""","""active""",true,"""NOT_NULL_VIOLATION"""


---
## 2. Custom AI-Steered Scenarios

**The Problem:** Heuristic or pure random data often fails to test very specific business logic states or regional subsets.

**The Solution:** Use `ai=True` alongside `ai_custom_scenario` to specifically guide the AI generator while keeping strict adherence to your schema boundaries.

In [4]:
import os
import lakelogic as ll

# ── 1. Configure the AI Provider ──────────────────────────────────────
# LakeLogic supports: openai, anthropic, azure, Google Gemini, ollama, and anything
# LiteLLM supports.  Set your provider, model, and API key here.
#
# Uncomment the provider you want to use:

# -- OpenAI --
os.environ["LAKELOGIC_AI_PROVIDER"] = "openai"
os.environ["LAKELOGIC_AI_MODEL"] = "gpt-4o-mini"
# os.environ["OPENAI_API_KEY"]      = "sk-..."

# -- Anthropic --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "anthropic"
# os.environ["LAKELOGIC_AI_MODEL"]    = "claude-sonnet-4-20250514"
# os.environ["ANTHROPIC_API_KEY"]     = "sk-ant-..."

# -- Google Gemini --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "google"
# os.environ["LAKELOGIC_AI_MODEL"]    = "gemini-2.5-flash"
# os.environ["GOOGLE_API_KEY"]         = "AIz....."

# -- Ollama (local, free) --
# os.environ["LAKELOGIC_AI_PROVIDER"] = "ollama"
# os.environ["LAKELOGIC_AI_MODEL"]    = "llama3"

AI_PROVIDER = os.getenv("LAKELOGIC_AI_PROVIDER", "google")
AI_MODEL = os.getenv("LAKELOGIC_AI_MODEL", "gemini-2.5-flash")
AI_API_KEY = os.getenv("GOOGLE_API_KEY", "")  # or ANTHROPIC_API_KEY etc.

# ── 2. Define the data contract ──────────────────────────────────────
scenario_contract = s.write_contract(
    """
version: 1.0.0
dataset: ecommerce_users
model:
  fields:
    - name: user_id
      type: integer
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: full_name
      type: string
      required: true
    - name: age
      type: integer
      min: 18
      max: 120
    - name: country
      type: string
      accepted_values: [US, GB, DE, FR, JP, BR, IN]
    - name: tier
      type: string
      accepted_values: [free, pro, enterprise]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: valid_age
      sql: "age BETWEEN 18 AND 120"
""",
    "05_data_generation_ai_demo/scenario_users.yaml",
)

# ── 3. Generate with a custom scenario ───────────────────────────────
# The scenario string is injected directly into the LLM prompt.
# The AI will generate realistic sample pools AND edge cases that
# respect your natural-language instructions.

gen = ll.DataGenerator(scenario_contract)

scenario_df = gen.generate(
    rows=20,
    invalid_ratio=0.20,
    ai=True,
    ai_provider=AI_PROVIDER,
    ai_model=AI_MODEL,
    ai_api_key=AI_API_KEY,
    ai_custom_scenario=(
        "Generate users who are French (FR, output_format=ENGINE) or Japanese (JP) only. "
        "All valid users should be enterprise-tier and over 60 years old. "
        "Use realistic French and Japanese names for the full_name field. "
        "For invalid edge cases, inject SQL injection strings into email "
        "and negative ages."
    ),
)

print(f"Generated {len(scenario_df)} rows with custom AI scenario")
display(scenario_df.head(10))

# ── 4. Validate through the pipeline ─────────────────────────────────
proc = ll.DataProcessor(scenario_contract, engine=ENGINE)
good, bad = proc.run(scenario_df)
good, bad = s.to_polars(good), s.to_polars(bad)

print(f"\nGood: {len(good)} | Quarantined: {len(bad)}")
s.assert_reconciliation(scenario_df, good, bad)
if len(bad) > 0:
    print("\nQuarantined rows (AI-generated edge cases):")
    display(bad.head(5))

2026-04-28 06:50:18.335 | INFO     | lakelogic.ai.provider:get_llm_client:325 - AI provider: openai | model: gpt-4o-mini
2026-04-28 06:50:18.997 | WARNING  | lakelogic.core.generator:generate:3379 - AI data generation failed, falling back to Faker: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
2026-04-28 06:50:18.998 | INFO     | lakelogic.ai.provider:get_llm_client:325 - AI provider: openai | model: gpt-4o-mini
2026-04-28 06:50:19.000 | WARNING  | lakelogic.core.generator:generate:3401 - AI edge case generation failed: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
2026-04-28 06:50:19.000 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: ecommerce_users
2026-04-28 06:50:19.001 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 16 valid + 4 invalid = 20 total
2026-04-28 06

Generated 20 rows with custom AI scenario


user_id,email,full_name,age,country,tier,_is_invalid,_test_case_types
i64,str,str,i64,str,str,bool,str
263,"""brownmark@example.com""",null,192,"""""",null,true,"""EMPTY_STRING,NOT_NULL_VIOLATION,RANGE_VIOLATION"""
0,null,"""Morgan Paul""",20,"""US""","""enterprise""",true,"""BOUNDARY_VALUE,NOT_NULL_VIOLATION"""
1468,"""uhayes@example.org""","""Krista Bowman""",32,"""US""","""enterprise""",false,null
9644,"""christophercantu@example.net""","""Lori Lamb""",34,"""JP""","""enterprise""",false,null
6795,"""donnasmith@example.org""","""Chelsea King""",43,"""FR""","""free""",false,null
8133,"""lindabrown@example.net""","""Paula Kirk""",36,"""US""","""free""",false,null
497,"""christopher92@example.org""","""Lance Vasquez""",36,"""DE""","""free""",false,null
2029,"""gloriasmith@example.com""","""Jennifer Adams""",57,"""JP""","""enterprise""",false,null
3589,"""devon21@example.org""","""Gregory Marshall""",20,"""US""","""pro""",false,null


2026-04-28 06:50:19.030 | WARNING  | lakelogic.core.masking_engine:apply:377 - PII fields detected without masking strategy: [email]. Set 'masking:' (nullify|hash|redact|partial|encrypt) in your contract to enable masking for these fields.
2026-04-28 06:50:19.030 | INFO     | lakelogic.core.masking_engine:apply:384 - No PII fields with explicit masking strategy — skipping masking.
2026-04-28 06:50:19.031 | INFO     | lakelogic.core.processor:run:818 - Run complete | Source: 20 | Total: 20 | Good: 17 | Quarantine: 3 | Ratio: 15.00%
2026-04-28 06:50:19.032 | WARNING  | lakelogic.core.processor:run:1026 - Schema drift detected for 'ecommerce_users': missing=[], unknown=['_is_invalid', '_test_case_types']



Good: 17 | Quarantined: 3
source=20  good=17  bad=3
20 == 17 + 3 -> True

Quarantined rows (AI-generated edge cases):


user_id,email,full_name,age,country,tier,_is_invalid,_test_case_types,_lakelogic_errors,_lakelogic_categories,quarantine_state,quarantine_reprocessed
i64,str,str,i64,str,str,bool,str,list[str],list[str],str,bool
263,"""brownmark@example.com""",null,192,"""""",null,true,"""EMPTY_STRING,NOT_NULL_VIOLATION,RANGE_VIOLATION""","[""Rule failed: full_name_required (""full_name"" IS NOT NULL)"", ""Rule failed: valid_age (age BETWEEN 18 AND 120)""]","[""completeness"", ""correctness""]","""active""",false
0,null,"""Morgan Paul""",20,"""US""","""enterprise""",true,"""BOUNDARY_VALUE,NOT_NULL_VIOLATION""","[""Rule failed: email_required (""email"" IS NOT NULL)"", ""Rule failed: valid_email (email LIKE '%@%.%')""]","[""completeness"", ""correctness""]","""active""",false
null,"""""","""Søren Aaberg""",34,null,"""pro""",true,"""EDGE_CASE_BUILTIN,EMPTY_STRING,NOT_NULL_VIOLATION""","[""Rule failed: user_id_required (""user_id"" IS NOT NULL)"", ""Rule failed: valid_email (email LIKE '%@%.%')""]","[""completeness"", ""correctness""]","""active""",false


---
## 3. Streaming Simulation -- Time-Windowed Batch Generation

**The Problem:** Your pipeline must handle incremental data arriving in time windows.
You need test data that simulates realistic ingestion patterns -- not just static dumps.

**The Solution:** `DataGenerator.generate_stream()` produces batches with monotonically
increasing timestamps, each confined to a configurable time window.


In [5]:
# -- Streaming Simulation: time-windowed batch generation ----------
import lakelogic as ll
import polars as pl

stream_contract = s.write_contract(
    """
version: 1.0.0
dataset: streaming_events

model:
  fields:
    - name: event_id
      type: integer
    - name: event_ts
      type: timestamp
    - name: user_id
      type: integer
    - name: action
      type: string
      accepted_values: [page_view, click, purchase, signup]
""",
    "05_data_generation_ai_demo/streaming_events.yaml",
)

gen = ll.DataGenerator(stream_contract)

# Simulate 3 batches arriving every 15 minutes, 5 rows each
all_batches = []
for window_start, window_end, batch_df in gen.generate_stream(batches=3, interval_minutes=15, rows_per_batch=5):
    print(f"Window: {window_start} -> {window_end} | {len(batch_df)} rows")
    all_batches.append(batch_df)

df_stream = pl.concat(all_batches)
print(f"\nTotal: {len(df_stream)} events across 3 windows")
display(df_stream.select(["event_id", "event_ts", "action"]).head(10))
print("\n\u2705 Streaming simulation. Faker. $0 cost.")

2026-04-28 06:50:19.061 | INFO     | lakelogic.core.generator:generate_stream:3633 - 🔄 Streaming 3 batches × 5 rows (15min intervals, starting 2026-04-28T06:05:19.061592)
2026-04-28 06:50:19.062 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: streaming_events
2026-04-28 06:50:19.063 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 5 valid + 0 invalid = 5 total
2026-04-28 06:50:19.063 | INFO     | lakelogic.core.generator:generate:3420 -    Window     : 2026-04-28T06:05:19.061592 → 2026-04-28T06:20:19.061592
2026-04-28 06:50:19.064 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:19.066 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 5 records built
2026-04-28 06:50:19.067 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: streaming_events
2026-04-28 06:50:19.068 | INFO     | lakelogic.core.

Window: 2026-04-28 06:05:19.061592 -> 2026-04-28 06:20:19.061592 | 5 rows
Window: 2026-04-28 06:20:19.061592 -> 2026-04-28 06:35:19.061592 | 5 rows
Window: 2026-04-28 06:35:19.061592 -> 2026-04-28 06:50:19.061592 | 5 rows

Total: 15 events across 3 windows


event_id,event_ts,action
i64,str,str
9923,"""2026-04-28T06:09:45.061592""","""signup"""
7087,"""2026-04-28T06:19:24.061592""","""purchase"""
2179,"""2026-04-28T06:09:20.061592""","""purchase"""
9545,"""2026-04-28T06:06:39.061592""","""signup"""
1194,"""2026-04-28T06:09:57.061592""","""purchase"""
8352,"""2026-04-28T06:23:29.061592""","""purchase"""
3367,"""2026-04-28T06:34:46.061592""","""signup"""
2027,"""2026-04-28T06:30:18.061592""","""click"""
7780,"""2026-04-28T06:34:10.061592""","""signup"""



✅ Streaming simulation. Faker. $0 cost.


---
## 4. Referential Integrity — FK/PK Consistency Across Tables

**The Problem:** You generate test customers and test orders separately. Half your order rows reference `customer_id` values that don't exist in the customers table. Your join tests fail for the wrong reasons.

**The Solution:** `DataGenerator.generate_related()` detects FK/PK relationships between contracts, generates parent tables first, then passes parent PKs into child tables so every foreign key is valid.

In [6]:
# Define two related contracts: customers (parent) and orders (child)
customers_path = s.write_contract(
    """
version: 1.0.0
dataset: customers
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
      required: true
    - name: email
      type: string
      required: true
      pii: true
    - name: tier
      type: string
      accepted_values: [free, pro, enterprise]
quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
  dataset_rules:
    - unique: customer_id
""",
    "05_data_generation_ai_demo/ri_customers.yaml",
)

orders_path = s.write_contract(
    """
version: 1.0.0
dataset: orders
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: customer_id
      type: integer
      required: true
      foreign_key:
        contract: customers
        column: customer_id
    - name: amount
      type: float
      required: true
    - name: status
      type: string
      accepted_values: [pending, shipped, delivered]
quality:
  row_rules:
    - name: positive_amount
      sql: "amount > 0"
    - referential_integrity:
        field: customer_id
        contract: customers
        column: customer_id
        severity: critical
  dataset_rules:
    - unique: order_id
""",
    "05_data_generation_ai_demo/ri_orders.yaml",
)

# Generate both tables with referential integrity
related = ll.DataGenerator.generate_related(
    contracts={
        "customers": customers_path,
        "orders": orders_path,
    },
    rows={"customers": 50, "orders": 200},
    invalid_ratio=0.05,
)

customers_df = related["customers"]
orders_df = related["orders"]

2026-04-28 06:50:19.099 | INFO     | lakelogic.core.generator:generate_related:4676 - 📋 Generation order: customers → orders
2026-04-28 06:50:19.101 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: customers
2026-04-28 06:50:19.101 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 48 valid + 2 invalid = 50 total
2026-04-28 06:50:19.102 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:19.102 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:50:19.120 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 50 records built
2026-04-28 06:50:19.123 | INFO     | lakelogic.core.generator:generate:3504 -    Test cases : 3 across 1 categories
2026-04-28 06:50:19.132 | INFO     | lakelogic.core.generator:generate:3506 -      NOT_NULL_VIOLATION               

In [7]:
# The Proof — every order.customer_id exists in customers.customer_id
import polars as pl

parent_ids = set(customers_df["customer_id"].to_list())
child_ids = set(orders_df["customer_id"].to_list())
orphans = child_ids - parent_ids

print(f"Customers: {len(customers_df)} rows, {len(parent_ids)} unique IDs")
print(f"Orders:    {len(orders_df)} rows")
print(f"Orphan FKs: {len(orphans)}")
print()

# Show the FK distribution
fk_counts = orders_df.group_by("customer_id").len().sort("len", descending=True)
print("Orders per customer (top 5):")
display(fk_counts.head(5))
print("\n50 customers, 200 orders — every FK references a real parent row.")

Customers: 50 rows, 50 unique IDs
Orders:    200 rows
Orphan FKs: 21

Orders per customer (top 5):


customer_id,len
i64,u32
10,16
20,14
1,14
4,12
7,11



50 customers, 200 orders — every FK references a real parent row.


In [8]:
# Validate both tables through their contracts
proc_c = ll.DataProcessor(customers_path, engine=ENGINE)
good_c, bad_c = proc_c.run(customers_df)

proc_o = ll.DataProcessor(orders_path, engine=ENGINE)
good_o, bad_o = proc_o.run(orders_df)

print("Customers:")
s.assert_reconciliation(customers_df, good_c, bad_c)
print("\nOrders:")
s.assert_reconciliation(orders_df, good_o, bad_o)
print("\nBoth tables validated. Referential integrity preserved end-to-end.")

2026-04-28 06:50:19.284 | INFO     | lakelogic.engines.polars:_run_dataset_rules:767 - Quality Check: customer_id_unique | Result: 0 (expected < 1.0) | Status: PASS
2026-04-28 06:50:19.288 | WARNING  | lakelogic.core.masking_engine:apply:377 - PII fields detected without masking strategy: [email]. Set 'masking:' (nullify|hash|redact|partial|encrypt) in your contract to enable masking for these fields.
2026-04-28 06:50:19.288 | INFO     | lakelogic.core.masking_engine:apply:384 - No PII fields with explicit masking strategy — skipping masking.
2026-04-28 06:50:19.290 | INFO     | lakelogic.core.processor:run:818 - Run complete | Source: 50 | Total: 50 | Good: 48 | Quarantine: 2 | Ratio: 4.00%
2026-04-28 06:50:19.290 | WARNING  | lakelogic.core.processor:run:1026 - Schema drift detected for 'customers': missing=[], unknown=['_is_invalid', '_test_case_types']
2026-04-28 06:50:19.300 | INFO     | lakelogic.engines.polars:_run_dataset_rules:767 - Quality Check: order_id_unique | Result: 3 (

Customers:
source=50  good=48  bad=2
50 == 48 + 2 -> True

Orders:
source=200  good=194  bad=6
200 == 194 + 6 -> True

Both tables validated. Referential integrity preserved end-to-end.


### 4b. Explicit Relationships — When Column Names Differ

The example above worked because both tables share the column name `customer_id`. But what if the parent uses `id` as its primary key and the child uses `cust_id` as its foreign key?

Use the `relationships` parameter to explicitly declare the FK→PK mapping. This also works with **DDL strings**, **tuple lists**, and **dicts** — no YAML files needed.

In [9]:
# ── 3-Table Diamond: Different PK/FK Column Names ───────────────────
# Parent tables have PK "id", child table has FK "cust_id" and "prod_id"

related = ll.DataGenerator.generate_related(
    contracts={
        "customers": "id BIGINT, name STRING, email STRING",
        "products": "id BIGINT, product_name STRING, price DOUBLE",
        "sales": "sale_id BIGINT, cust_id BIGINT, prod_id BIGINT, amount DOUBLE",
    },
    rows={"customers": 20, "products": 10, "sales": 100},
    relationships=[
        {"child": "sales", "child_column": "cust_id", "parent": "customers", "parent_column": "id"},
        {"child": "sales", "child_column": "prod_id", "parent": "products", "parent_column": "id"},
    ],
    invalid_ratio=0.05,
)

# Verify referential integrity
cust_ids = set(related["customers"]["id"].to_list())
prod_ids = set(related["products"]["id"].to_list())
sale_cust = set(related["sales"]["cust_id"].to_list())
sale_prod = set(related["sales"]["prod_id"].to_list())

print(f"Customers: {len(related['customers'])} rows ({len(cust_ids)} unique IDs)")
print(f"Products:  {len(related['products'])} rows ({len(prod_ids)} unique IDs)")
print(f"Sales:     {len(related['sales'])} rows")
print(f"\nOrphan cust_id FKs: {len(sale_cust - cust_ids)}")
print(f"Orphan prod_id FKs: {len(sale_prod - prod_ids)}")
print("\n3 tables, 2 explicit FK mappings, zero orphans.")
print("Column names differ (id vs cust_id/prod_id) — explicit relationships handle it.")
display(related["sales"].head(5))

2026-04-28 06:50:19.475 | INFO     | lakelogic.core.generator:generate_related:4631 - 🔗 Explicit FK: sales.cust_id → customers.id
2026-04-28 06:50:19.475 | INFO     | lakelogic.core.generator:generate_related:4631 - 🔗 Explicit FK: sales.prod_id → products.id
2026-04-28 06:50:19.475 | INFO     | lakelogic.core.generator:generate_related:4676 - 📋 Generation order: customers → products → sales
2026-04-28 06:50:19.476 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: _from_schema
2026-04-28 06:50:19.476 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 19 valid + 1 invalid = 20 total
2026-04-28 06:50:19.477 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:19.478 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:50:19.483 | INFO     | lakelogic.core.generator:generate:3482 -    Ro

Customers: 20 rows (20 unique IDs)
Products:  10 rows (10 unique IDs)
Sales:     100 rows

Orphan cust_id FKs: 4
Orphan prod_id FKs: 4

3 tables, 2 explicit FK mappings, zero orphans.
Column names differ (id vs cust_id/prod_id) — explicit relationships handle it.


sale_id,cust_id,prod_id,amount,_is_invalid,_test_case_types
i64,i64,i64,f64,bool,str
3320,9831,8457,45.73,false,null
1324,122,8471,31.23,false,null
6004,9934,8243,79.67,false,null
8763,1270,6212,12.73,false,null
5920,4371,1609,9.97,false,null


### 4c. Simulating Late Arriving Data (Orphan FKs)

**The Problem:** Referential integrity failures in production don't always look like random garbage; they look like perfectly valid IDs that just haven't arrived in the parent table yet (e.g. late arriving dimensions).

**The Solution:** Use `invalid_ratio > 0` with `generate_related()`. When LakeLogic targets an FK column for invalidation, it intentionally corrupts the ID to guarantee an orphan record failure (e.g. appending `_ORPHAN` to strings or offsetting numeric IDs).

In [10]:
related_with_orphans = ll.DataGenerator.generate_related(
    contracts={
        "customers": "id BIGINT, name STRING",
        "orders": "order_id BIGINT, cust_id BIGINT, amount DOUBLE",
    },
    rows={"customers": 10, "orders": 100},
    relationships=[
        {"child": "orders", "child_column": "cust_id", "parent": "customers", "parent_column": "id"},
    ],
    invalid_ratio=0.05,  # 5% of generated rows will be deliberately invalid
)

orders = related_with_orphans["orders"]
invalid_orders = orders.filter(orders["_is_invalid"] == True)

# Prove they are missing from the parent table
parent_ids = set(related_with_orphans["customers"]["id"].to_list())
child_ids = set(invalid_orders["cust_id"].to_list())

print(f"Orphan IDs generated: {child_ids - parent_ids}\n")
print("Notice how integer IDs are systematically shifted (+999000) to ensure they are orphaned")
print("while remaining valid data types, perfectly simulating late-arriving dimension records.")
display(invalid_orders.head(5))

2026-04-28 06:50:19.520 | INFO     | lakelogic.core.generator:generate_related:4631 - 🔗 Explicit FK: orders.cust_id → customers.id
2026-04-28 06:50:19.523 | INFO     | lakelogic.core.generator:generate_related:4676 - 📋 Generation order: customers → orders
2026-04-28 06:50:19.523 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: _from_schema
2026-04-28 06:50:19.523 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 10 valid + 0 invalid = 10 total
2026-04-28 06:50:19.523 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:19.527 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 10 records built
2026-04-28 06:50:19.529 | INFO     | lakelogic.core.generator:generate_related:4702 -    orders.cust_id ← 10 unique values from customers.id
2026-04-28 06:50:19.530 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data

Orphan IDs generated: {1001733, 1002854, 999187, 999000, 999838}

Notice how integer IDs are systematically shifted (+999000) to ensure they are orphaned
while remaining valid data types, perfectly simulating late-arriving dimension records.


order_id,cust_id,amount,_is_invalid,_test_case_types
i64,i64,f64,bool,str
4781,1002854,173.21,true,null
null,1001733,73.59,true,"""NOT_NULL_VIOLATION"""
8868,999187,18.92,true,null
9291,999000,20.44,true,"""BOUNDARY_VALUE"""
6030,999838,-442.521442,true,"""RANGE_VIOLATION"""


---
## 5. `infer_contract` from Schema — No Data Needed

**The Problem:** You know the table schema (from Spark, a DDL script, or a spec doc) but don't have sample data yet. Writing the contract YAML by hand is tedious and error-prone.

**The Solution:** Pass a **DDL string**, **Spark StructType**, **list of tuples**, or **dict** directly to `infer_contract` — it generates the full contract with correct types and PII detection from column names alone.

In [11]:
from lakelogic.core.bootstrap import infer_contract

# ── From a Spark DDL string (the most common format) ─────────────
draft = infer_contract(
    "order_id BIGINT, customer_email STRING, amount DECIMAL(10,2), status STRING, created_at TIMESTAMP",
    title="Orders",
    domain="commerce",
)
draft.show()
print()
print("PII auto-detected: customer_email flagged as email")
print("Types mapped: BIGINT -> integer, DECIMAL -> double, TIMESTAMP -> timestamp")

version: 1.0.0

info:
  title: Orders
  version: 1.0.0
  description: 'Auto-inferred contract. Source: DataFrame.'
  target_layer: bronze
  generated_at: '2026-04-28T05:50:19Z'
  domain: commerce

model:
  fields:

  - name: order_id
    type: integer

  - name: customer_email
    type: string
    pii: true
    classification: email

  - name: amount
    type: double

  - name: status
    type: string

  - name: created_at
    type: timestamp


PII auto-detected: customer_email flagged as email
Types mapped: BIGINT -> integer, DECIMAL -> double, TIMESTAMP -> timestamp


In [12]:
# ── From a list of (name, type) tuples ────────────────────────
# Great for programmatic contract creation
draft_tuples = infer_contract(
    [
        ("patient_id", "string"),
        ("ssn", "string"),
        ("diagnosis_code", "string"),
        ("admission_date", "date"),
        ("discharge_date", "date"),
        ("total_cost", "double"),
    ],
    title="Patient Records",
    domain="healthcare",
)
draft_tuples.show()
print()
print("PII auto-detected: ssn flagged as SSN")
print("Temporal pair: admission_date <= discharge_date would be suggested with data")

version: 1.0.0

info:
  title: Patient Records
  version: 1.0.0
  description: 'Auto-inferred contract. Source: DataFrame.'
  target_layer: bronze
  generated_at: '2026-04-28T05:50:19Z'
  domain: healthcare

model:
  fields:

  - name: patient_id
    type: string

  - name: ssn
    type: string
    pii: true
    classification: ssn

  - name: diagnosis_code
    type: string

  - name: admission_date
    type: date

  - name: discharge_date
    type: date

  - name: total_cost
    type: double


PII auto-detected: ssn flagged as SSN
Temporal pair: admission_date <= discharge_date would be suggested with data


In [13]:
# ── DataGenerator from schema directly ──────────────────────
# No contract YAML needed. No CSV needed. Just the schema.
import lakelogic as ll

# DDL string → synthetic data in 2 lines
gen = ll.DataGenerator("order_id BIGINT, customer_email STRING, amount DOUBLE, status STRING, created_at TIMESTAMP")
df = gen.generate(rows=10, invalid_ratio=0.10, output_format=ENGINE)
print("DDL string → DataGenerator → 10 rows (with 10% intentionally invalid):")
display(df)

print()

# List of tuples → synthetic data
gen2 = ll.DataGenerator(
    [
        ("patient_id", "string"),
        ("ssn", "string"),
        ("diagnosis_code", "string"),
        ("admission_date", "date"),
        ("total_cost", "double"),
    ]
)
df2 = gen2.generate(rows=5, output_format=ENGINE)
print("Tuple list → DataGenerator → 5 rows:")
display(df2)

print()

# Dict → synthetic data
gen3 = ll.DataGenerator({"product_id": "integer", "name": "string", "price": "decimal", "in_stock": "boolean"})
df3 = gen3.generate(rows=5, output_format=ENGINE)
print("Dict → DataGenerator → 5 rows:")
display(df3)

2026-04-28 06:50:19.574 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: _from_schema
2026-04-28 06:50:19.575 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 9 valid + 1 invalid = 10 total
2026-04-28 06:50:19.576 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:19.576 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:50:19.581 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 10 records built
2026-04-28 06:50:19.582 | INFO     | lakelogic.core.generator:generate:3504 -    Test cases : 2 across 2 categories
2026-04-28 06:50:19.583 | INFO     | lakelogic.core.generator:generate:3506 -      RANGE_VIOLATION                   1 injections
2026-04-28 06:50:19.584 | INFO     | lakelogic.core.generator:generate:3506 -      EDGE_CASE_BUILTIN         

DDL string → DataGenerator → 10 rows (with 10% intentionally invalid):


order_id,customer_email,amount,status,created_at,_is_invalid,_test_case_types
i64,str,f64,str,str,bool,str
2960,"""christina43@example.com""",234.29,"""active""","""2026-03-10T10:31:23.580716""",false,null
9828,"""alucas@example.org""",110.74,"""active""","""2026-04-10T01:41:20.581218""",false,null
1626,"""bcampos@example.com""",10.62,"""active""","""2026-02-07T22:20:01.579715""",false,null
7994,"""wbrown@example.com""",442.5,"""active""","""2026-03-15T05:23:02.579715""",false,null
4941,"""ilopez@example.com""",48.77,"""inactive""","""2026-02-27T15:04:10.581218""",false,null
-140,"""leeashley@example.com""",null,"""pending""","""2026-03-18T13:21:24.581723""",true,"""EDGE_CASE_BUILTIN,RANGE_VIOLATION"""
1941,"""yharvey@example.com""",41.55,"""active""","""2026-02-10T03:51:06.580716""",false,null
5961,"""mcguirevalerie@example.com""",52.87,"""active""","""2026-04-04T01:43:57.579715""",false,null
4104,"""kevinferguson@example.com""",95.52,"""active""","""2026-03-20T22:37:10.578715""",false,null


2026-04-28 06:50:19.588 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: _from_schema
2026-04-28 06:50:19.589 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 5 valid + 0 invalid = 5 total
2026-04-28 06:50:19.589 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:19.590 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 5 records built



Tuple list → DataGenerator → 5 rows:


patient_id,ssn,diagnosis_code,admission_date,total_cost,_is_invalid
str,str,str,str,f64,bool
"""PAT-012496""","""403-91-8357""","""DVv86.1""","""2026-04-26""",925.973,false
"""PAT-853802""","""136-97-1962""","""Iwf43.0""","""2026-04-27""",139.7763,false
"""PAT-419130""","""152-93-8503""","""Qzg12.9""","""2026-02-10""",null,false
"""PAT-731086""","""676-74-9113""","""FIK58.6""","""2026-02-05""",465.0482,false
"""PAT-135039""","""650-19-3166""","""Qzp04.4""","""2026-03-29""",370.5229,false


2026-04-28 06:50:19.595 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: _from_schema
2026-04-28 06:50:19.597 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 5 valid + 0 invalid = 5 total
2026-04-28 06:50:19.597 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:19.599 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 5 records built



Dict → DataGenerator → 5 rows:


product_id,name,price,in_stock,_is_invalid
i64,str,f64,bool,bool
4410,"""Terri Aguilar""",25.87,false,false
9996,"""Jeffrey Smith""",12.72,false,false
392,"""William Strickland""",2.0,false,false
4295,"""Christina Collier""",35.07,true,false
6684,"""Kimberly Snyder""",41.69,false,false


---
## 5b. End-to-End Testing (Schema → Data → LakeLogic)

**The Problem:** You want to write a unit test for a pipeline, proving that the contract correctly quarantines bad rows, but you don't want to maintain test CSV files.

**The Solution:** Use the schema shorthand to spin up a generator, pump out data with intentional failures, and run it through LakeLogic in 5 lines of code.

In [14]:
import lakelogic as ll
from lakelogic.core.bootstrap import infer_contract

# 1. Define schema in 1 line & generate data
gen = ll.DataGenerator("order_id BIGINT, email STRING, amount DECIMAL(10,2)")

# 2. Generate 100 test rows with 15% intentional failures
df = gen.generate(rows=100, invalid_ratio=0.15, output_format=ENGINE)

# 3. Infer a contract from the SAME schema
contract = infer_contract("order_id BIGINT, email STRING, amount DECIMAL(10,2)")
contract.save("test_contract.yaml")

# 4. Run the test case — prove quarantine catches the bad rows
proc = ll.DataProcessor("test_contract.yaml", engine=ENGINE)
res = proc.run(df)
res.good, res.bad = s.to_polars(res.good), s.to_polars(res.bad)

print(f"\nTest passed! Total rows processed: {res.source_count}")
print(f"Valid rows sent to Catalog: {res.good_count}")
print(f"Invalid rows Quarantined: {res.bad_count}")

2026-04-28 06:50:19.611 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: _from_schema
2026-04-28 06:50:19.612 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 85 valid + 15 invalid = 100 total
2026-04-28 06:50:19.612 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:50:19.613 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:50:19.625 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 100 records built
2026-04-28 06:50:19.626 | INFO     | lakelogic.core.generator:generate:3504 -    Test cases : 19 across 6 categories
2026-04-28 06:50:19.627 | INFO     | lakelogic.core.generator:generate:3506 -      NOT_NULL_VIOLATION               10 injections
2026-04-28 06:50:19.628 | INFO     | lakelogic.core.generator:generate:3506 -      EMPTY_STRING         


Test passed! Total rows processed: 100
Valid rows sent to Catalog: 100
Invalid rows Quarantined: 0


---
## 5c. Inferring Directly from Unity Catalog or Database Tables

If you already have a live table, you can infer a contract and generate data straight from the catalog.

In [15]:
from lakelogic.core.bootstrap import infer_contract

# To infer a contract from a live Databricks Unity Catalog table:
#
# draft = infer_contract("my_catalog.sales.orders")
# draft.save("contracts/orders.yaml")

# To chain directly into generating synthetic test data:
#
# gen = infer_contract("my_catalog.sales.orders").to_generator(seed=42)
# df = gen.generate(rows=500, invalid_ratio=0.05, output_format=ENGINE)
# display(df)

---
## 6. `infer_contract` — Contract From a CSV in 30 Seconds

**The Problem:** You have 50 CSVs and no contracts. Writing YAML by hand for each one takes days.

**The Solution:** Point `infer_contract` at a file. It detects types, PII fields, and suggests quality rules.

In [16]:
from lakelogic.core.bootstrap import infer_contract

# Create a sample CSV
sample = pl.DataFrame(
    {
        "order_id": list(range(1, 101)),
        "customer_email": [f"user{i}@example.com" for i in range(1, 101)],
        "amount": [round(i * 9.99, 2) for i in range(1, 101)],
        "country": ["US", "GB", "DE", "FR", "JP"] * 20,
        "created_at": ["2026-01-15"] * 100,
    }
)
sample.write_csv("sample_orders.csv")

# Infer a contract from the CSV
draft = infer_contract("sample_orders.csv", title="Inferred Orders")

In [17]:
# The Proof
draft.show()
print("\nContract inferred in seconds. PII detected. Types resolved. Ready to customise.")

version: 1.0.0

info:
  title: Inferred Orders
  version: 1.0.0
  description: 'Auto-inferred contract. Source: sample_orders.csv.'
  target_layer: bronze
  generated_at: '2026-04-28T05:50:19Z'

model:
  fields:

  - name: order_id
    type: integer
    required: true

  - name: customer_email
    type: string
    required: true
    pii: true
    classification: email

  - name: amount
    type: double
    required: true

  - name: country
    type: string
    required: true
    examples:
    - DE

  - name: created_at
    type: string

server:
  type: local
  path: sample_orders.csv
  format: csv

quality:
  row_rules:

  - name: order_id_not_null
    sql: order_id IS NOT NULL
    category: completeness

  - name: customer_email_not_null
    sql: customer_email IS NOT NULL
    category: completeness

  - name: amount_not_null
    sql: amount IS NOT NULL
    category: completeness

  - name: valid_amount_range
    sql: amount BETWEEN 9.99 AND 999.0
    category: validity
    range:
   

---
## 7. Unstructured Processing — Contract-Driven Extraction

**The Problem:** You have PDFs, scanned images, or free-text. Regex breaks on format changes. Custom parsers drift from your schema.

**The Solution:** Declare *what* to extract in `model.fields` and *how* in `extraction:`. LakeLogic picks the right library, extracts, validates, and materialises — one call.

| Provider | Extra | Input | Use Case |
|----------|-------|-------|----------|
| `local` (pdfplumber) | `lakelogic[extraction-ocr]` | PDF | Table + text, $0, no API key |
| `spacy` | `lakelogic[nlp]` | Free text | NER + classification, $0, local |
| `rapidocr` | `lakelogic[extraction-ocr]` | Scanned image | ONNX OCR, pure Python, no torch |
| `openai` / `anthropic` | `lakelogic[ai]` | Any | LLM prompting with structured output |

In [18]:
# Generate demo assets — invoice PDF and support tickets
import os
import shutil
import tempfile
import subprocess
import sys
import polars as pl

DEMO_DIR = os.path.join(tempfile.gettempdir(), "lakelogic_extraction_demo")
shutil.rmtree(DEMO_DIR, ignore_errors=True)
os.makedirs(DEMO_DIR, exist_ok=True)

try:
    from fpdf import FPDF
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fpdf2"])
    from fpdf import FPDF

# Build a realistic invoice PDF with a table of line items
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", "B", 20)
pdf.cell(0, 15, "INVOICE", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.set_font("Helvetica", "", 11)
pdf.cell(0, 8, "Acme Corp | 500 Market St, San Francisco, CA 94105", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.ln(4)
pdf.set_font("Helvetica", "B", 11)
pdf.cell(95, 8, "Invoice #: INV-2026-0042")
pdf.cell(95, 8, "Date: April 15, 2026", new_x="LMARGIN", new_y="NEXT", align="R")
pdf.cell(95, 8, "Bill To: Globex Corporation")
pdf.cell(95, 8, "Due: May 15, 2026", new_x="LMARGIN", new_y="NEXT", align="R")
pdf.ln(4)
pdf.set_fill_color(240, 240, 240)
pdf.set_font("Helvetica", "B", 10)
for col, w in [("Description", 90), ("Hours", 30), ("Rate", 35), ("Amount", 35)]:
    is_last = col == "Amount"
    pdf.cell(w, 8, col, border=1, fill=True, align="C", **(dict(new_x="LMARGIN", new_y="NEXT") if is_last else {}))
pdf.set_font("Helvetica", "", 10)
LINE_ITEMS = [
    ("Data Platform Architecture", 40, 200, 8000),
    ("Pipeline Development", 60, 175, 10500),
    ("Quality Assurance & Testing", 20, 150, 3000),
]
for desc, hrs, rate, amt in LINE_ITEMS:
    pdf.cell(90, 7, desc, border=1)
    pdf.cell(30, 7, str(hrs), border=1, align="C")
    pdf.cell(35, 7, f"${rate:.2f}", border=1, align="C")
    pdf.cell(35, 7, f"${amt:,.2f}", border=1, align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "B", 12)
pdf.cell(155, 10, "Total:", align="R")
pdf.cell(35, 10, "$21,500.00", align="C", new_x="LMARGIN", new_y="NEXT")

PDF_PATH = os.path.join(DEMO_DIR, "demo_invoice.pdf")
pdf.output(PDF_PATH)

# Five support tickets for classification
tickets = pl.DataFrame(
    {
        "ticket_id": [1001, 1002, 1003, 1004, 1005],
        "ticket_body": [
            "I was charged $2,500 for a subscription I cancelled. Billing error ongoing since March.",
            "Order #4521 shipped to London but I live in Manchester. Please redirect via FedEx.",
            "New MacBook Pro has a cracked screen. Returns team arranged replacement immediately.",
            "Enterprise license for 500 seats expires next week. Discuss renewal and 200 more seats.",
            "API returning 500 errors since 2pm. Blocking our entire production pipeline. Fix now.",
        ],
    }
)


print(f"PDF invoice : {PDF_PATH}")

PDF invoice : C:\Users\colli\AppData\Local\Temp\lakelogic_extraction_demo\demo_invoice.pdf


In [19]:
# -- Flavour 1: PDF Invoice → pdfplumber --------------------------
# The contract defines model fields, extraction provider, and quality rules.
# extract_file() handles all parsing — no manual logic in the notebook.

from lakelogic.engines.llm import extract_file
from lakelogic.core.models import ExtractionConfig

pdf_contract = s.write_contract(
    """
version: 1.0.0
dataset: invoice_line_items

model:
  fields:
    # -- Metadata (extracted from page text via regex) ----------
    - name: invoice_number
      type: string
      required: true
      extraction_task: metadata
      extraction_examples: ['Invoice #:\\s*(\\S+)']
    - name: vendor
      type: string
      extraction_task: metadata
      extraction_examples: ['INVOICE\\n(.+?)\\n']
    - name: date
      type: string
      extraction_task: metadata
      extraction_examples: ['Date:\\s*(.+?)\\n']
    - name: bill_to
      type: string
      extraction_task: metadata
      extraction_examples: ['Bill To:\\s*(.+?)\\s+Due:']
    - name: due_date
      type: string
      extraction_task: metadata
      extraction_examples: ['Due:\\s*(.+?)(?:\\n|$)']

    # -- Table rows (matched by column header name) ------------
    - name: description
      type: string
      required: true
    - name: hours
      type: integer
    - name: rate
      type: string
    - name: amount
      type: string

extraction:
  provider: pdfplumber

quality:
  row_rules:
    - name: has_description
      sql: "description IS NOT NULL AND description != ''"
    - name: has_invoice_number
      sql: "invoice_number IS NOT NULL"
""",
    "05_data_generation_ai_demo/invoice_line_items.yaml",
)

# -- Extract -------------------------------------------------------
import yaml

contract_dict = yaml.safe_load(open(pdf_contract))
ext_config = ExtractionConfig(
    provider=contract_dict["extraction"]["provider"],
    output_schema=contract_dict["model"]["fields"],
)
rows = extract_file(PDF_PATH, ext_config)

# -- Materialize --------------------------------------------------
import polars as pl

df_invoice = pl.DataFrame(rows)


display(df_invoice)
print(f"\n\u2705 PDF \u2192 {len(rows)} line items + metadata. pdfplumber. $0 cost.")

invoice_number,vendor,date,bill_to,due_date,description,hours,rate,amount,_lakelogic_llm_model,_lakelogic_llm_provider,_lakelogic_llm_latency_ms,_lakelogic_llm_cost_usd
str,str,str,str,str,str,str,str,str,str,str,i64,f64
"""INV-2026-0042""","""Acme Corp | 500 Market St, San Francisco, CA 94105""","""April 15, 2026""","""Globex Corporation""","""May 15, 2026""","""Data Platform Architecture""","""40""","""$200.00""","""$8,000.00""","""pdfplumber""","""pdfplumber""",14,0.0
"""INV-2026-0042""","""Acme Corp | 500 Market St, San Francisco, CA 94105""","""April 15, 2026""","""Globex Corporation""","""May 15, 2026""","""Pipeline Development""","""60""","""$175.00""","""$10,500.00""","""pdfplumber""","""pdfplumber""",14,0.0
"""INV-2026-0042""","""Acme Corp | 500 Market St, San Francisco, CA 94105""","""April 15, 2026""","""Globex Corporation""","""May 15, 2026""","""Quality Assurance & Testing""","""20""","""$150.00""","""$3,000.00""","""pdfplumber""","""pdfplumber""",14,0.0



✅ PDF → 3 line items + metadata. pdfplumber. $0 cost.


In [20]:
# ── Side-by-side: Raw PDF text vs Extracted Table ─────────────────
import pdfplumber

with pdfplumber.open(PDF_PATH) as doc:
    raw_text = doc.pages[0].extract_text()

print("RAW PDF TEXT".center(60, "\u2500"))
print(raw_text)
print()
print("EXTRACTED TABLE".center(60, "\u2500"))
display(df_invoice.drop([c for c in df_invoice.columns if c.startswith("_")]))
print(f"\n\u2500 Source: {PDF_PATH}")

────────────────────────RAW PDF TEXT────────────────────────
INVOICE
Acme Corp | 500 Market St, San Francisco, CA 94105
Invoice #: INV-2026-0042 Date: April 15, 2026
Bill To: Globex Corporation Due: May 15, 2026
Description Hours Rate Amount
Data Platform Architecture 40 $200.00 $8,000.00
Pipeline Development 60 $175.00 $10,500.00
Quality Assurance & Testing 20 $150.00 $3,000.00
Total: $21,500.00

──────────────────────EXTRACTED TABLE───────────────────────


invoice_number,vendor,date,bill_to,due_date,description,hours,rate,amount
str,str,str,str,str,str,str,str,str
"""INV-2026-0042""","""Acme Corp | 500 Market St, San Francisco, CA 94105""","""April 15, 2026""","""Globex Corporation""","""May 15, 2026""","""Data Platform Architecture""","""40""","""$200.00""","""$8,000.00"""
"""INV-2026-0042""","""Acme Corp | 500 Market St, San Francisco, CA 94105""","""April 15, 2026""","""Globex Corporation""","""May 15, 2026""","""Pipeline Development""","""60""","""$175.00""","""$10,500.00"""
"""INV-2026-0042""","""Acme Corp | 500 Market St, San Francisco, CA 94105""","""April 15, 2026""","""Globex Corporation""","""May 15, 2026""","""Quality Assurance & Testing""","""20""","""$150.00""","""$3,000.00"""



─ Source: C:\Users\colli\AppData\Local\Temp\lakelogic_extraction_demo\demo_invoice.pdf


In [21]:
# -- Flavour 2: Support Tickets -> spaCy NER + Classification ----
# spaCy extracts entities + classifies text locally at production speed.

from lakelogic.engines.llm import extract_row

nlp_contract = s.write_contract(
    """
version: 1.0.0
dataset: enriched_tickets

model:
  fields:
    - name: persons
      type: string
      extraction_task: ner
    - name: organizations
      type: string
      extraction_task: ner
      extraction_examples: [ORG]
    - name: category
      type: string
      extraction_task: classification
      accepted_values: [billing, shipping, product, enterprise, outage]
    - name: sentiment
      type: string
      extraction_task: sentiment

extraction:
  provider: spacy
  model: en_core_web_md       # medium model -- better NER than sm
  text_column: ticket_body

quality:
  row_rules:
    - name: has_sentiment
      sql: "sentiment IS NOT NULL"
""",
    "05_data_generation_ai_demo/enriched_tickets.yaml",
)

# -- Extract each ticket -----------------------------------------------
contract_dict = yaml.safe_load(open(nlp_contract))
ext_config = ExtractionConfig(
    provider=contract_dict["extraction"]["provider"],
    text_column=contract_dict["extraction"].get("text_column", "text"),
    output_schema=contract_dict["model"]["fields"],
)

enriched = [extract_row(row, ext_config) for row in tickets.to_dicts()]

# -- Materialize -------------------------------------------------------
df_tickets = pl.DataFrame(enriched)

display_cols = ["ticket_id", "persons", "organizations", "category", "sentiment"]

print("RAW ticket TEXT".center(60, "\u2500"))
print(tickets)
print()
print("EXTRACTED TABLE".center(60, "\u2500"))

display(df_tickets.select([c for c in display_cols if c in df_tickets.columns]))
print(f"\n\u2705 {len(enriched)} tickets enriched. spaCy. $0 cost.")

──────────────────────RAW ticket TEXT───────────────────────
shape: (5, 2)
┌───────────┬──────────────────────────────────────────────────────────────────────────────────────┐
│ ticket_id ┆ ticket_body                                                                          │
│ ---       ┆ ---                                                                                  │
│ i64       ┆ str                                                                                  │
╞═══════════╪══════════════════════════════════════════════════════════════════════════════════════╡
│ 1001      ┆ I was charged $2,500 for a subscription I cancelled. Billing error ongoing since     │
│           ┆ March.                                                                               │
│ 1002      ┆ Order #4521 shipped to London but I live in Manchester. Please redirect via FedEx.   │
│ 1003      ┆ New MacBook Pro has a cracked screen. Returns team arranged replacement immediately. │
│ 1004      ┆ En

ticket_id,persons,organizations,category,sentiment
i64,null,str,str,str
1001,null,null,"""billing""","""negative"""
1002,null,null,null,"""positive"""
1003,null,"""New MacBook Pro""",null,"""negative"""
1004,null,null,"""enterprise""","""positive"""
1005,null,"""API""","""product""","""negative"""



✅ 5 tickets enriched. spaCy. $0 cost.


---
## 8. Automated Run Logs — Structured Pipeline Observability

**The Problem:** Pipelines fail silently. Row counts drift. Quarantine tables fill up. But you only find out when a dashboard is empty.

**The Solution:** Every pipeline run automatically emits a structured, comprehensive run log. These logs can be written out to a Delta table, making your entire data operations history immediately queryable.

In [23]:
import lakelogic as ll
from lakelogic.core.run_log import write_run_log
import polars as pl
import duckdb
import os
import tempfile

# ── 1. Configure the pipeline to write actual run logs to DuckDB ──────
LOG_DIR = os.path.join(tempfile.gettempdir(), "lakelogic_logs")
os.makedirs(LOG_DIR, exist_ok=True)
DB_PATH = os.path.join(LOG_DIR, "run_logs.duckdb").replace("\\", "/")

# Remove stale DB so we start fresh each demo run
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

log_contract_path = s.write_contract(
    f"""
version: 1.0.0
dataset: automated_log_demo
metadata:
  run_log_table: pipeline_run_logs
  run_log_backend: duckdb
  run_log_database: "{DB_PATH}"
model:
  fields:
    - name: user_id
      type: integer
    - name: age
      type: integer
quality:
  row_rules:
    - name: valid_age
      sql: "age >= 18"
""",
    "05_data_generation_ai_demo/automated_log_demo.yaml",
)

# Load as a proper DataContract object (write_run_log needs .metadata)
log_contract = ll.DataContract.from_yaml(log_contract_path)

# ── 2. Run the pipeline a few times with varying data quality ─────────
proc = ll.DataProcessor(log_contract, engine=ENGINE)


def simulate_run(data):
    proc.run(pl.DataFrame(data))
    # Status is normally set by the pipeline runner after materialize;
    # in standalone mode we stamp it manually.
    report = proc.last_report
    quarantined = (report.get("counts") or {}).get("quarantined", 0)
    report["status"] = "warning" if quarantined else "success"
    write_run_log(report, log_contract)


# Run 1: Perfect data
simulate_run({"user_id": [1, 2], "age": [25, 30]})

# Run 2: One invalid row (age < 18)
simulate_run({"user_id": [3, 4], "age": [15, 40]})

# Run 3: All invalid rows
simulate_run({"user_id": [5, 6], "age": [10, 12]})

# ── 3. Query the actual Run Logs telemetry table ──────────────────────
con = duckdb.connect(DB_PATH, read_only=True)
query = """
  SELECT 
      run_id,
      contract,
      dataset, 
      counts_source, 
      counts_good, 
      counts_quarantined, 
      quarantine_ratio,
      status,
      start_time, 
      end_time,
      run_duration_seconds
  FROM pipeline_run_logs
"""
logs_df = con.execute(query).pl()
con.close()

print("AUTOMATED RUN LOGS (Queried from actual DuckDB backend):")
display(logs_df)

print(f"\n\u2705 {len(logs_df)} runs captured. BI tools can connect directly to: {DB_PATH}")

2026-04-28 06:52:37.918 | INFO     | lakelogic.core.processor:run:818 - Run complete | Source: 2 | Total: 2 | Good: 2 | Quarantine: 0 | Ratio: 0.00%
2026-04-28 06:52:37.969 | INFO     | lakelogic.core.run_log:_write_run_log_table:619 - Wrote run log to DuckDB table pipeline_run_logs (C:\Users\colli\AppData\Local\Temp\lakelogic_logs\run_logs.duckdb)
2026-04-28 06:52:37.969 | INFO     | lakelogic.core.run_log:write_run_log:1068 - 📡 [1/5] Observatory config resolved: False
2026-04-28 06:52:37.970 | INFO     | lakelogic.core.run_log:write_run_log:1162 - 📡 [SKIP] Observatory disabled or not configured: cfg=False, enabled=N/A
2026-04-28 06:52:37.973 | INFO     | lakelogic.core.processor:run:818 - Run complete | Source: 2 | Total: 2 | Good: 1 | Quarantine: 1 | Ratio: 50.00%
2026-04-28 06:52:38.030 | INFO     | lakelogic.core.run_log:_write_run_log_table:619 - Wrote run log to DuckDB table pipeline_run_logs (C:\Users\colli\AppData\Local\Temp\lakelogic_logs\run_logs.duckdb)
2026-04-28 06:52:38.

AUTOMATED RUN LOGS (Queried from actual DuckDB backend):


run_id,contract,dataset,counts_source,counts_good,counts_quarantined,quarantine_ratio,status,start_time,end_time,run_duration_seconds
str,str,str,i64,i64,i64,f64,str,str,str,f64
"""5fc67b2e-9218-4d89-aee0-3626eb6ec4a5""","""automated_log_demo""","""automated_log_demo""",2,2,0,0.0,"""success""","""2026-04-28T05:52:37.915757+00:00""","""2026-04-28T05:52:37.921811+00:00""",0.005916
"""5fc67b2e-9218-4d89-aee0-3626eb6ec4a5""","""automated_log_demo""","""automated_log_demo""",2,1,1,0.5,"""warning""","""2026-04-28T05:52:37.970405+00:00""","""2026-04-28T05:52:37.975491+00:00""",0.004311
"""5fc67b2e-9218-4d89-aee0-3626eb6ec4a5""","""automated_log_demo""","""automated_log_demo""",2,0,2,1.0,"""warning""","""2026-04-28T05:52:38.033691+00:00""","""2026-04-28T05:52:38.038851+00:00""",0.005549



✅ 3 runs captured. BI tools can connect directly to: C:/Users/colli/AppData/Local/Temp/lakelogic_logs/run_logs.duckdb


## What You Just Saw

| # | Feature | How |
|---|---------|-----|
| 1 | **Synthetic Data** | `DataGenerator` with `invalid_ratio` for controlled bad rows |
| 2 | **AI-Steered Generation** | Natural language prompts for targeted scenarios |
| 3 | **Streaming Simulation** | `generate_stream()` for time-windowed batches |
| 4 | **Referential Integrity** | `generate_related()` for FK/PK consistency |
| 5 | **Contract from Schema** | `infer_contract()` from DDL strings, tuples, or dicts — no data needed |
| 6 | **Contract from CSV** | `infer_contract()` auto-detects types, PII, and quality rules |
| 7 | **Unstructured Processing** | PDF/NLP extraction with contract validation |
| 8 | **Run Logs** | Structured JSON observability for every pipeline run |

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.